In [ ]:
from google.colab import drive
drive.mount('/content/drive')

#!pip install -q sentencepiece transformers datasets accelerate

Mounted at /content/drive


In [3]:
from google.colab import files
uploaded = files.upload()

Saving intent_analysis_test.csv to intent_analysis_test.csv
Saving intent_analysis_train.csv to intent_analysis_train.csv
Saving intent_analysis_val.csv to intent_analysis_val.csv


In [4]:
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, set_seed

set_seed(42)

train_df = pd.read_csv('intent_analysis_train.csv')
val_df = pd.read_csv('intent_analysis_val.csv')

label_map = {"similar_items": 0, "graph_pairing": 1, "color_variant": 2}

for df in [train_df, val_df]:
    df['intent'] = df['intent'].str.strip()
    df['labels'] = df['intent'].map(label_map)
    df.dropna(subset=['labels'], inplace=True)
    df['labels'] = df['labels'].astype(int)

train_dataset = Dataset.from_pandas(train_df)
eval_dataset = Dataset.from_pandas(val_df)

MODEL_ID = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

def tokenize_function(examples):
    return tokenizer(examples["query"], padding="max_length", truncation=True, max_length=32)

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_eval = eval_dataset.map(tokenize_function, batched=True)

tokenized_train = tokenized_train.remove_columns(["query", "intent"])
tokenized_eval = tokenized_eval.remove_columns(["query", "intent"])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/2806 [00:00<?, ? examples/s]

Map:   0%|          | 0/312 [00:00<?, ? examples/s]

In [5]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
from sklearn.metrics import classification_report
import numpy as np

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID,
    num_labels=3,
    id2label={v: k for k, v in label_map.items()},
    label2id=label_map
)

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    lr_scheduler_type="linear",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.05,
    warmup_steps=125,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=True,
    report_to="none"
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)
    target_names = ["similar_items", "graph_pairing", "color_variant"]
    report = classification_report(labels, predictions, target_names=target_names, output_dict=True, zero_division=0)
    return {
        "accuracy": report["accuracy"],
        "f1_similar": report["similar_items"]["f1-score"],
        "f1_color": report["color_variant"]["f1-score"]
    }

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print('training started')
trainer.train()
print('training completed')

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


training started


Epoch,Training Loss,Validation Loss,Accuracy,F1 Similar,F1 Color
1,No log,0.045358,0.993590,1.000000,0.991071
2,No log,0.035367,0.993590,1.000000,0.991071
3,0.197462,0.038907,0.993590,1.000000,0.991071
4,0.197462,0.026370,0.996795,1.000000,0.995556
5,0.197462,0.027036,0.996795,1.000000,0.995556


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

training completed


In [6]:
test_df = pd.read_csv('intent_analysis_test.csv')
test_df['labels'] = test_df['intent'].map(label_map)
test_dataset = Dataset.from_pandas(test_df)

tokenized_test = test_dataset.map(tokenize_function, batched=True)
tokenized_test = tokenized_test.remove_columns(["query", "intent"])

predictions = trainer.predict(tokenized_test)
preds = np.argmax(predictions.predictions, axis=-1)
labels = predictions.label_ids

print(classification_report(labels, preds, target_names=["similar_items", "graph_pairing", "color_variant"], digits=4))

Map:   0%|          | 0/225 [00:00<?, ? examples/s]

               precision    recall  f1-score   support

similar_items     1.0000    1.0000    1.0000        74
graph_pairing     1.0000    1.0000    1.0000        76
color_variant     1.0000    1.0000    1.0000        75

     accuracy                         1.0000       225
    macro avg     1.0000    1.0000    1.0000       225
 weighted avg     1.0000    1.0000    1.0000       225



In [7]:
import os

FINAL_SAVE_PATH = "/content/drive/MyDrive/model_cache/intent_classifier_roberta_vfinal"
os.makedirs(FINAL_SAVE_PATH, exist_ok=True)

trainer.save_model(FINAL_SAVE_PATH)
tokenizer.save_pretrained(FINAL_SAVE_PATH)
print("model saved")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

model saved


In [ ]:
import pandas as pd
import torch
from transformers import pipeline

MODEL_PATH = "/content/drive/MyDrive/model_cache/intent_classifier_roberta_vfinal"

device = 0 if torch.cuda.is_available() else -1
classifier = pipeline("text-classification", model=MODEL_PATH, tokenizer=MODEL_PATH, device=device)

test_cases = [
    #chứa từ 'blue' (color) nhưng ý định chính là tìm đồ phối cùng
    {"query": "I love this blue dress, what shoes would match it?", "expected": "graph_pairing"},

    # chứa cụm 'wear with' (pairing) và 'black' (color) nhưng ý định là tìm hàng 
    {"query": "need a dupe for this bag to wear with my black dress", "expected": "similar_items"},

    # chứa cụm 'match my boots' (pairing) nhưng ý định chính là đổi màu áo
    {"query": "I need this exact same shirt in black to match my boots", "expected": "color_variant"},

    # câu dài, diễn đạt phức tạp, không dùng từ khóa chuẩn như 'similar'
    {"query": "got anything that looks exactly like this but isn't as expensive?", "expected": "similar_items"},

    # không dùng tên màu sắc cụ thể, dùng biệt ngữ ngành thời trang 'darker wash'
    {"query": "do you have this specific one in a darker wash?", "expected": "color_variant"},

    # chứa 'exact same' (color variant) và 'blue', nhưng thực tế là tìm phụ kiện
    {"query": "accessories to go with these exact same blue jeans", "expected": "graph_pairing"},

    # chứa 'wear for' nhưng thực tế là tìm đồ thay thế
    {"query": "find me a cheaper alternative to this to wear for a wedding", "expected": "similar_items"},

    # câu hỏi có chứa ngữ cảnh thời gian quá khứ
    {"query": "is there a red version of the jacket I just bought?", "expected": "color_variant"},

    # câu cụt, không dùng giới từ chuẩn như 'with' hay 'for'
    {"query": "what kind of belt fits this?", "expected": "graph_pairing"},

    # ý định ẩn, hoàn toàn không có từ khóa nhận diện trực tiếp
    {"query": "this is too pricey, show me something like it", "expected": "similar_items"}
]

print("Running adversarial inference...\n")

results = []
for case in test_cases:
    pred = classifier(case["query"], truncation=True, max_length=32)[0]
    results.append({
        "Query": case["query"],
        "Expected": case["expected"],
        "Predicted": pred["label"],
        "Confidence": f"{pred['score']:.4f}"
    })

df_results = pd.DataFrame(results)
df_results['Pass'] = df_results['Expected'] == df_results['Predicted']

# Căn chỉnh hiển thị terminal
pd.set_option('display.max_colwidth', None)
print(df_results.to_string(index=False))

print(f"\nAccuracy on Adversarial Set: {df_results['Pass'].sum()}/10")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Running adversarial inference...

                                                            Query      Expected     Predicted Confidence  Pass
               I love this blue dress, what shoes would match it? graph_pairing graph_pairing     0.9998  True
             need a dupe for this bag to wear with my black dress similar_items similar_items     0.9998  True
          I need this exact same shirt in black to match my boots color_variant color_variant     0.9998  True
got anything that looks exactly like this but isn't as expensive? similar_items similar_items     0.9998  True
                  do you have this specific one in a darker wash? color_variant color_variant     0.9998  True
               accessories to go with these exact same blue jeans graph_pairing graph_pairing     0.9998  True
      find me a cheaper alternative to this to wear for a wedding similar_items similar_items     0.9998  True
              is there a red version of the jacket I just bought? color_varian